# 11. Dựng đồ thị NF-BoT-IoT-v3 (task 1.8, GĐ B)

Bộ dữ liệu thứ 2 (IoT) theo đúng pipeline của bộ chính: cửa sổ 5 s (W = S), node là IP, chia theo khối (blocked), cân theo số node, gap 1 cửa sổ, scaler fit trên train, không lấy mẫu flow.

**NF-BoT-IoT-v3 không có lớp Botnet_Backdoor** (và không có Khac). Các lớp có mặt: Normal, DoS_DDoS (DoS + DDoS), Truy_cap_trai_phep (Reconnaissance + Theft). Ngưỡng 100/30/30 node chỉ xét các lớp có mặt.

Các bước: (1) pytest, (2) đối chiếu sha256, số dòng, số flow từng nhãn, (3) `split_search.py` chọn số khối nhỏ nhất đạt tiêu chí, (4) `build_dataset_cli.py`, (5) `verify_graphs.py`, (6) `run_info.json`.

Kaggle CPU (không bật GPU), bật Internet, gắn dataset `athena21/netflow-v3-datasets`. Output `processed/nf_bot_iot_v3/` (`.pt`, `split.json`, `scaler.pkl`) ở lại trên Kaggle cho GĐ C–H.

In [ ]:
import glob, hashlib, json, os, resource, subprocess, sys, time
from contextlib import contextmanager

WORK = "/kaggle/working"
REPO = "/tmp/gnn-iot-ids"          # ngoài /kaggle/working để không lẫn vào output
INFO = {"steps_s": {}, "device": "CPU", "gpu": None}
T_START = time.time()


def peak_ram_gb():
    """RAM đỉnh (GB) của tiến trình notebook và mọi tiến trình con đã kết thúc (ru_maxrss tính bằng KB trên Linux)."""
    kb = max(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss, resource.getrusage(resource.RUSAGE_CHILDREN).ru_maxrss)
    return round(kb / 2**20, 2)


def save_info():
    INFO["total_s"] = round(time.time() - T_START, 1)
    INFO["peak_ram_gb"] = peak_ram_gb()
    with open(f"{WORK}/run_info.json", "w", encoding="utf-8") as f:
        json.dump(INFO, f, ensure_ascii=False, indent=1)


@contextmanager
def step(name):
    t0 = time.time()
    try:
        yield
    finally:
        INFO["steps_s"][name] = round(time.time() - t0, 1)
        save_info()                # ghi sau mỗi bước, lỗi giữa chừng vẫn còn run_info.json
        print(f"--- {name}: {INFO['steps_s'][name]} s, RAM đỉnh tới nay {INFO['peak_ram_gb']} GB")


def run(cmd, ok=(0,), **kw):
    r = subprocess.run(cmd, capture_output=True, text=True, **kw)
    print(r.stdout[-7000:], end="")
    print(r.stderr[-2500:], end="", file=sys.stderr)
    print(f"[mã thoát {r.returncode}]")
    if r.returncode not in ok:
        raise RuntimeError(f"Lệnh lỗi: {' '.join(cmd)}")
    return r


with step("clone_va_cai_goi"):
    run(["rm", "-rf", REPO])
    run(["git", "clone", "--depth", "1", "https://github.com/minucedev/gnn-iot-ids.git", REPO])
    run(["git", "-C", REPO, "log", "-1", "--format=%H %s"])
    INFO["commit"] = subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
    with open(f"{REPO}/requirements.txt") as f:
        reqs = [l.strip() for l in f if l.strip() and not l.startswith("#")]
    keep = [r for r in reqs if r.split("==")[0].lower() not in {"torch", "jupyter", "ipykernel"}]
    run([sys.executable, "-m", "pip", "install", "-q", *keep])

try:
    import torch
    if torch.cuda.is_available():
        INFO["device"], INFO["gpu"] = "GPU", torch.cuda.get_device_name(0)
except ImportError:
    pass
INFO["cpu_count"] = os.cpu_count()
print("Commit:", INFO["commit"], "| máy:", INFO["device"], INFO["gpu"], "| CPU:", INFO["cpu_count"])

## 1. pytest (dừng nếu có test lỗi)

In [ ]:
with step("pytest"):
    r = run([sys.executable, "-m", "pytest", "-q"], cwd=REPO)      # mã thoát khác 0 thì run() ném lỗi, notebook dừng
    INFO["pytest"] = r.stdout.strip().splitlines()[-1]

## 2. Đối chiếu file gốc: sha256, số dòng, số flow từng nhãn

In [ ]:
EXPECT = {
    "size": 3_817_527_532,
    "sha256": "8bde1f6f1c8bc59dcb49828fb5b9d65c0b63e06d92b2b9f15b37159e923009ea",
    "rows": 16_933_808,
    "labels": {"DoS": 8_034_190, "DDoS": 7_150_882, "Reconnaissance": 1_695_132, "Theft": 1_615, "Benign": 51_989},
}
CSV = next(glob.iglob("/kaggle/input/**/NF-BoT-IoT-v3.csv", recursive=True))
print("CSV:", CSV)

with step("doi_chieu_du_lieu"):
    import pyarrow.csv as pacsv
    size = os.path.getsize(CSV)
    h = hashlib.sha256()
    with open(CSV, "rb") as f:
        for blk in iter(lambda: f.read(64 * 2**20), b""):
            h.update(blk)
    attack = pacsv.read_csv(CSV, convert_options=pacsv.ConvertOptions(include_columns=["Attack"])).column("Attack").to_pandas()
    got = {"size": size, "sha256": h.hexdigest(), "rows": int(len(attack)),
           "labels": {k: int(v) for k, v in attack.value_counts().items()}}
    del attack
    checks = {"size": got["size"] == EXPECT["size"], "sha256": got["sha256"] == EXPECT["sha256"],
              "rows": got["rows"] == EXPECT["rows"], "labels": got["labels"] == EXPECT["labels"]}
    out = {"expected": EXPECT, "got": got, "match": checks}
    with open(f"{WORK}/data_check_nf_bot_iot_v3.json", "w", encoding="utf-8") as f:
        json.dump(out, f, ensure_ascii=False, indent=1)
    print(json.dumps(out, ensure_ascii=False, indent=1))
    INFO["data_check"] = checks
    assert all(checks.values()), f"Dữ liệu không khớp số đã kiểm ở task 1.2: {checks}"

## 3. Chọn số khối bằng `split_search.py`

Chỉ xét cấu hình `blocked` chia theo số node. Chọn số khối **nhỏ nhất** đạt cả hai tiêu chí (mỗi lớp có mặt ≥ 100/30/30 node và tỉ lệ node 70/15/15 ± 3). Nếu không cấu hình nào đạt thì **không hạ ngưỡng**: lấy cấu hình ít lớp hụt nhất để ghi đúng số node, đánh dấu không đạt, và báo người chạy.

In [ ]:
SS = f"{WORK}/split_search_nf_bot_iot_v3.json"
with step("split_search"):
    run([sys.executable, "-I", "scripts/split_search.py", "nf_bot_iot_v3", CSV, "5", "5", SS], cwd=REPO)

res = json.load(open(SS, encoding="utf-8"))
print("Lớp có mặt:", res["classes"], "| vắng mặt:", res["absent_classes"])
cand = [c for c in res["configs"] if c["mode"] == "blocked" and c["balance_nodes"]]
print(f"{'khối':>5} {'train/val/test %':>20}  lớp hụt ngưỡng")
for c in sorted(cand, key=lambda c: c["blocks"]):
    r_ = c["ratio_pct"]
    print(f"{c['blocks']:5d} {r_['train']:6.1f}/{r_['val']:4.1f}/{r_['test']:4.1f} {'tỉ lệ đạt' if c['ok_ratio'] else 'tỉ lệ lệch':>12}  {c['missing_classes'] or 'đủ'}")
assert cand, "split_search không cho cấu hình blocked nào dùng được"
passing = [c for c in cand if c["ok_classes"] and c["ok_ratio"]]
if passing:
    pick = min(passing, key=lambda c: c["blocks"])
else:
    pick = min(cand, key=lambda c: (len(c["missing_classes"]), not c["ok_ratio"], c["blocks"]))
BLOCKS = pick["blocks"]
INFO.update(blocks=BLOCKS, split_search_passed=bool(passing), absent_classes=res["absent_classes"],
            picked_counts=pick["counts"])
save_info()
print(f"\n==> Chọn blocks = {BLOCKS}", "(ĐẠT tiêu chí)" if passing else "(KHÔNG cấu hình nào đạt: ngưỡng giữ nguyên, ghi số node thật, báo người chạy)")
print(json.dumps(pick["counts"], ensure_ascii=False, indent=1))

## 4. Dựng đồ thị chính thức

In [ ]:
OUT = f"{WORK}/processed/nf_bot_iot_v3"
with step("build_dataset"):
    # mã 3 = có kiểm tra không đạt: vẫn giữ output để xem summary.json, báo ở cuối
    r = run([sys.executable, "-I", "scripts/build_dataset_cli.py", "nf_bot_iot_v3", CSV, "5", "5", "blocked", "1", OUT,
             f"blocks={BLOCKS}", "balance=1", "tag=nf_bot_iot_v3_w5s5_blocked"], ok=(0, 3), cwd=REPO)
    INFO["build_exit_code"] = r.returncode

## 5. Kiểm tra lại đồ thị đã lưu

In [ ]:
with step("verify_graphs"):
    r = run([sys.executable, "-I", "scripts/verify_graphs.py", OUT, f"{WORK}/verify_nf_bot_iot_v3.json"], ok=(0, 3), cwd=REPO)
    INFO["verify_exit_code"] = r.returncode

## 6. Tổng kết và run_info.json

In [ ]:
summary = json.load(open(f"{OUT}/summary.json", encoding="utf-8"))
verify = json.load(open(f"{WORK}/verify_nf_bot_iot_v3.json", encoding="utf-8"))
print("Hash:", summary["fingerprint_sha256"], "| số đồ thị:", summary["n_graphs"], "| tùy chọn:", summary["options"])
print("Số node mỗi lớp × mỗi tập:")
for part, row in summary["split_counts"].items():
    print(f"  {part:5s}", row)
print("\nKiểm tra khi dựng:")
for c in summary["checks"]:
    print(f"  [{'ĐẠT' if c['ok'] else 'KHÔNG ĐẠT'}] {c['name']} ({c['detail']})")
print("Kiểm tra lại (verify_graphs):")
for c in verify:
    print(f"  [{'ĐẠT' if c['ok'] else 'KHÔNG ĐẠT'}] {c['name']} ({c['detail']})")
INFO["all_checks_pass"] = all(c["ok"] for c in summary["checks"]) and all(c["ok"] for c in verify)
INFO["n_graphs"] = summary["n_graphs"]
INFO["fingerprint_sha256"] = summary["fingerprint_sha256"]
save_info()
print("\nrun_info.json:")
print(open(f"{WORK}/run_info.json", encoding="utf-8").read())
run(["du", "-sh", f"{WORK}/processed"])
if not (INFO["all_checks_pass"] and INFO["split_search_passed"]):
    print("CHÚ Ý: có tiêu chí không đạt, xem các dòng KHÔNG ĐẠT ở trên.")